In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, pipeline, set_seed
from torch.utils.data import DataLoader
from datasets import load_dataset
from pprint import pprint

In [19]:
pretrained_model_path = "/Users/zhangyf/llm/gpt2-chinese-cluecorpussmall"
dataset = load_dataset("csv", data_files="online_shopping_10_cats.csv")

In [20]:
dataset["train"][0]

{'cat': '书籍',
 'label': 1,
 'review': '\ufeff做父母一定要有刘墉这样的心态，不断地学习，不断地进步，不断地给自己补充新鲜血液，让自己保持一颗年轻的心。我想，这是他能很好的和孩子沟通的一个重要因素。读刘墉的文章，总能让我看到一个快乐的平易近人的父亲，他始终站在和孩子同样的高度，给孩子创造着一个充满爱和自由的生活环境。很喜欢刘墉在字里行间流露出的做父母的那种小狡黠，让人总是忍俊不禁，父母和子女之间有时候也是一种战斗，武力争斗过于低级了，智力较量才更有趣味。所以，做父母的得加把劲了，老思想老观念注定会一败涂地，生命不息，学习不止。家庭教育，真的是乐在其中。'}

In [21]:
ds_train = dataset["train"]
# 将评论少于1024个字的过滤出来
ds_train = ds_train.filter(lambda x: x["review"] != None and len(
    x["review"]) > 20 and len(x["review"]) < 1024)

In [22]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_path)
model = AutoModelForCausalLM.from_pretrained(pretrained_model_path)

In [23]:
tokenizer.pad_token

'[PAD]'

In [24]:
total_parameters = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_parameters}")

Total parameters: 102068736


In [25]:
def tokenize(batch):
    return tokenizer(batch["review"])

In [26]:
map_kwargs = {
    "batched": True,
    "batch_size": 512,
    "remove_columns": ["cat", "label", "review"]
}

In [27]:
tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)

In [28]:
tokenized_dataset_train.set_format("torch")

In [29]:
# 将pad_token作为eos_token
tokenizer.eos_token = tokenizer.pad_token

In [30]:
tokenized_dataset_train[0]["input_ids"].shape

torch.Size([257])

In [31]:
tokenized_dataset_train[10]["input_ids"].shape

torch.Size([170])

In [32]:
# 将数据整理成预测下一个token的任务的数据格式
data_collator = DataCollatorForLanguageModeling(
    tokenizer,
    mlm=False  # 将数据整理成预测下一个token的格式
)

In [33]:
dataloader_params = {
    "batch_size": 2,
    "collate_fn": data_collator
}

train_dataloader = DataLoader(
    tokenized_dataset_train,
    **dataloader_params
)

In [34]:
# 要更新的是model的参数
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
# 一般sft会训练1个epoch，也就是把训练数据看一遍就可以了
# 否则容易过拟合，造成灾难性遗忘
num_epochs = 1
# device = "mps" if torch.mps.is_available() else "cpu"
device = "cpu"

In [35]:
PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0

In [36]:
model.to(device)
for epoch in range(num_epochs):
    model.train()
    for i, batch in enumerate(train_dataloader):
        batch = batch.to(device)
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if i % 100 == 0:
            print(f"Epoch: {epoch}, Step: {i}, Loss: {loss.item()}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch: 0, Step: 0, Loss: 3.052706241607666
Epoch: 0, Step: 100, Loss: 2.672497510910034
Epoch: 0, Step: 200, Loss: 2.8058547973632812
Epoch: 0, Step: 300, Loss: 2.8143532276153564
Epoch: 0, Step: 400, Loss: 2.6361827850341797
Epoch: 0, Step: 500, Loss: 2.783541679382324
Epoch: 0, Step: 600, Loss: 2.9651224613189697
Epoch: 0, Step: 700, Loss: 2.9076268672943115
Epoch: 0, Step: 800, Loss: 3.0386834144592285
Epoch: 0, Step: 900, Loss: 2.648231267929077
Epoch: 0, Step: 1000, Loss: 2.388523578643799
Epoch: 0, Step: 1100, Loss: 2.8731002807617188
Epoch: 0, Step: 1200, Loss: 2.854384183883667
Epoch: 0, Step: 1300, Loss: 1.9243890047073364
Epoch: 0, Step: 1400, Loss: 2.732379913330078
Epoch: 0, Step: 1500, Loss: 2.5685482025146484
Epoch: 0, Step: 1600, Loss: 2.7133853435516357
Epoch: 0, Step: 1700, Loss: 2.934981346130371
Epoch: 0, Step: 1800, Loss: 2.9610042572021484
Epoch: 0, Step: 1900, Loss: 2.628350019454956
Epoch: 0, Step: 2000, Loss: 2.4709014892578125
Epoch: 0, Step: 2100, Loss: 3.3538

In [37]:
model.save_pretrained("./gpt2-sft")
tokenizer.save_pretrained("./gpt2-sft")

('./gpt2-sft/tokenizer_config.json',
 './gpt2-sft/special_tokens_map.json',
 './gpt2-sft/vocab.txt',
 './gpt2-sft/added_tokens.json',
 './gpt2-sft/tokenizer.json')

In [38]:
# 测试微调后的模型
g = pipeline("text-generation", model="/Users/zhangyf/llm/gpt2-sft")
set_seed(43)
pprint(g("这本书真是", max_length=300, num_return_sequences=1))

Device set to use mps:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=300) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '这本书真是这样的 他们, this is a movie that understands the value '
                    'of family.                     .                      '
                    '.                                                                                                                                                                                             '}]
